# P100 + Py3.10 (uv venv): SadTalker validation

In [ ]:
# 0. uv -> standalone Python 3.10 venv with P100-compatible torch (cu117, sm_60)
import subprocess, sys, os
VENV='/kaggle/working/venv310'
PY=VENV+'/bin/python'
subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
UV=os.popen('which uv').read().strip() or (sys.executable.replace('python','')+'uv')
print('uv', UV)
subprocess.run([UV,'python','install','3.10'],check=True)
subprocess.run([UV,'venv',VENV,'--python','3.10'],check=True)
def venv(cmd):
    print('$',cmd[:2])
    return subprocess.run([UV,'pip','install','--python',PY]+cmd,check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL).returncode
venv(['torch==2.0.1+cu117','torchvision==0.15.2+cu117','--index-url','https://download.pytorch.org/whl/cu117'])
for p in ['numpy==1.23.5','scipy==1.10.1','scikit-image==0.19.3','imageio==2.31.1','imageio-ffmpeg==0.4.7','pydub==0.25.1','resampy==0.3.1','joblib==1.2.0','librosa==0.9.2','numba==0.56.4','yacs==0.1.8','pyyaml','tqdm','av==10.0.1','safetensors','kornia==0.6.8','face_alignment==1.3.5','basicsr==1.4.2','facexlib==0.3.0','gfpgan','einops','opencv-python-headless','tensorboard','dlib']:
    venv([p])
# SadTalker/librosa still imports pkg_resources
subprocess.run([UV,'pip','install','--python',PY,'--reinstall','setuptools==68.2.2'],check=True)
# Final ABI lock: some SadTalker dependencies may upgrade NumPy; restore the compatible stack last
for p in ['numpy==1.23.5','scipy==1.10.1','scikit-image==0.19.3']:
    subprocess.run([UV,'pip','install','--python',PY,'--reinstall','--no-deps',p],check=True)
rc=subprocess.run([PY,'-c','import torch,numpy,skimage; print("TORCH",torch.__version__,"CUDA",torch.cuda.is_available(),"ARCH",torch.cuda.get_arch_list()); print("NUMPY",numpy.__version__,"SKIMAGE",skimage.__version__)'],capture_output=True,text=True)
print(rc.stdout); print(rc.stderr[:500])
print('venv ready')


In [ ]:
# 1. Profile + 30s Kokoro voice (system python)
import subprocess, sys, os
subprocess.run([sys.executable,'-m','pip','install','-q','gdown','kokoro','soundfile'],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
subprocess.run(['apt-get','-qq','install','-y','ffmpeg'],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
import imageio_ffmpeg, shutil
FF=imageio_ffmpeg.get_ffmpeg_exe()
if not os.path.exists('/usr/local/bin/ffmpeg'): shutil.copy(FF,'/usr/local/bin/ffmpeg'); os.chmod('/usr/local/bin/ffmpeg',0o755)
os.makedirs('/kaggle/working/test',exist_ok=True)
subprocess.run(['gdown','1-2sFUEHqXDbaPq0lfBmamrjQBsdL_QuY','-O','/kaggle/working/test/profile.jpg'],check=False)
print('profile', os.path.getsize('/kaggle/working/test/profile.jpg'))
from kokoro import KPipeline
import soundfile as sf, numpy as np
pipe=KPipeline(lang_code='a', device='cpu')
txt='My name is Mirsina Aghdam, CEO of EDGE, Earthwise Dynamics Geo Environs. We are an Irish company working in geoengineering, AI automation and critical-mineral intelligence. Europe needs secure rare-earth supplies, but exploration remains slow and fragmented.'
chunks=[]
for gs,ps,a in pipe(txt, voice='am_michael', speed=1.0): chunks.append(a.cpu().numpy())
a=np.concatenate(chunks) if chunks else np.zeros(24000*30)
a=a[:24000*30] if len(a)>=24000*30 else np.pad(a,(0,24000*30-len(a)))
sf.write('/kaggle/working/test/seg.wav', a.astype('float32'), 24000)
print('audio', os.path.getsize('/kaggle/working/test/seg.wav'))


In [ ]:
# 2. Clone SadTalker + download models
import os, subprocess
if not os.path.exists('/kaggle/working/SadTalker'):
    subprocess.run(['git','clone','https://github.com/OpenTalker/SadTalker.git','/kaggle/working/SadTalker'],check=True)
os.chdir('/kaggle/working/SadTalker')
r=subprocess.run(['bash','scripts/download_models.sh'],check=False)
print('models rc', r.returncode, 'present', os.path.exists('checkpoints/SadTalker_V0.0.2_512.safetensors'))


In [ ]:
# 3. Run SadTalker ONE clip via Py3.10 venv
import os, subprocess, glob, shutil
PY='/kaggle/working/venv310/bin/python'
os.chdir('/kaggle/working/SadTalker')
cmd=[PY,'inference.py','--driven_audio','/kaggle/working/test/seg.wav','--source_image','/kaggle/working/test/profile.jpg','--result_dir','/kaggle/working/sad_test','--size','512','--preprocess','crop','--still','--batch_size','1','--cpu']
print('running sadtalker...'); r=subprocess.run(cmd,check=False)
print('rc', r.returncode)
files=glob.glob('/kaggle/working/sad_test/*/*.mp4')
print('outputs', files)
if files:
    shutil.copy(files[0],'/kaggle/working/test/sad_clip.mp4'); print('copied', files[0])


In [ ]:
# 4. Report
import os, glob
print('SADTALKER clip:', glob.glob('/kaggle/working/test/sad_clip.mp4'))
print('AUDIO:', os.path.exists('/kaggle/working/test/seg.wav'))
print('PROFILE:', os.path.exists('/kaggle/working/test/profile.jpg'))
